# Diffusion Model (DDPM) — Butterfly Image Generation

Class-conditional DDPM to generate synthetic butterfly images for data augmentation.

**Structure:**
1. Setup & Imports
2. Data Loading
3. Training Function
4. Baseline DDPM
5. Grid Search
6. Final Training
7. Generate Augmented Dataset
8. Retrain Baseline CNN + Evaluate
9. Metrics & Comparison

## 1. Setup & Imports

In [1]:
import os, sys, json, random, itertools, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as T
import torchvision.utils as vutils
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score
from skimage.metrics import structural_similarity as ssim_metric

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchmetrics[image]', '-q'])
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore

import kagglehub

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():            device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else:                                    device = torch.device('cpu')
print('Device:', device)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
for c in [os.path.join(NOTEBOOK_DIR,'../src'), os.path.join(NOTEBOOK_DIR,'src'),
          '/Applications/Universidade/4ano_2semestre/ACA/projeto2/aml-butterfly-generative-augmentation/src']:
    c = os.path.abspath(c)
    if os.path.isdir(c) and c not in sys.path:
        sys.path.append(c); break

from dataset import ButterflyDataset
from transforms import get_transforms
from models import BaselineCNN
from utils import get_splits, get_class_mapping, GLOBAL_SEED
from diffusion import GaussianDiffusion, UNet, linear_beta_schedule, cosine_beta_schedule
print('OK')

Device: mps
OK


/Users/matildecarvalho/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Data Loading

In [2]:
path = kagglehub.competition_download('aca-tp-2')
train_dir = os.path.join(path, 'train')

df = pd.read_csv(os.path.join(path, 'train.csv'))[['filename','label']]
train_df, val_df = get_splits(df, seed=GLOBAL_SEED)
class_to_idx, idx_to_class, classes = get_class_mapping(df)
NUM_CLASSES = len(classes)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Classes: {NUM_CLASSES}')

IMAGE_SIZE = 64
BATCH_SIZE = 32

diff_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.ToTensor(),
    T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])
diff_transform_val = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

train_ds = ButterflyDataset(df=train_df, img_dir=train_dir, transform=diff_transform)
val_ds   = ButterflyDataset(df=val_df,   img_dir=train_dir, transform=diff_transform_val)
train_ld = data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_ld   = data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

def denorm(t): return (t * 0.5 + 0.5).clamp(0,1)
print('Data loaders ready.')

Train: 4159 | Val: 1040 | Classes: 75
Data loaders ready.


## 3. Training Function

In [3]:
def train_ddpm(timesteps=500, schedule='cosine', base_ch=64, lr=1e-4,
               epochs=60, patience=8, verbose=True):
    """Train a class-conditional DDPM on 64x64 butterfly images."""
    betas = cosine_beta_schedule(timesteps) if schedule == 'cosine' else linear_beta_schedule(timesteps)
    diffusion = GaussianDiffusion(betas).to(device)
    model     = UNet(base_ch=base_ch, num_classes=NUM_CLASSES).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/10)

    best_loss, no_imp, best_state = float('inf'), 0, None
    history = []

    for epoch in range(epochs):
        model.train(); ep_loss = 0
        for imgs, labels in train_ld:
            imgs = imgs.to(device); labels = labels.to(device)
            t    = torch.randint(0, timesteps, (imgs.size(0),), device=device)
            loss = diffusion.p_losses(model, imgs, t, class_labels=labels)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
        ep_loss /= len(train_ld)

        model.eval(); val_loss = 0
        with torch.no_grad():
            for imgs, labels in val_ld:
                imgs = imgs.to(device); labels = labels.to(device)
                t    = torch.randint(0, timesteps, (imgs.size(0),), device=device)
                val_loss += diffusion.p_losses(model, imgs, t, class_labels=labels).item()
        val_loss /= len(val_ld)
        scheduler.step()
        history.append({'epoch': epoch+1, 'train_loss': ep_loss, 'val_loss': val_loss})
        if verbose and (epoch+1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}/{epochs} | train={ep_loss:.4f} | val={val_loss:.4f}')

        if val_loss < best_loss:
            best_loss = val_loss; no_imp = 0
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= patience: break

    model.load_state_dict(best_state); model.to(device)
    return model, diffusion, best_loss, history

print('train_ddpm() defined.')

train_ddpm() defined.


## 4. Baseline DDPM (default hypers)

In [4]:
print('Training baseline DDPM (T=500, cosine, base_ch=64, lr=1e-4, 60 epochs)...')
model_base, diff_base, base_val_loss, base_hist = train_ddpm(
    timesteps=500, schedule='cosine', base_ch=64, lr=1e-4, epochs=60, patience=8)
print(f'Baseline DDPM best val loss: {base_val_loss:.4f}')

Training baseline DDPM (T=500, cosine, base_ch=64, lr=1e-4, 60 epochs)...
Epoch  10/60 | train=0.0723 | val=0.0657
Epoch  20/60 | train=0.0619 | val=0.0598


KeyboardInterrupt: 

In [ ]:
hist_df = pd.DataFrame(base_hist)
plt.figure(figsize=(8,4))
plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train')
plt.plot(hist_df['epoch'], hist_df['val_loss'],   label='Val')
plt.title('Baseline DDPM — Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/ddpm_baseline_loss.png', dpi=150); plt.show()

In [ ]:
# Preview: generate 16 images (one per class, first 16 classes)
model_base.eval()
preview_labels = torch.arange(min(16, NUM_CLASSES), device=device)
with torch.no_grad():
    samples = diff_base.p_sample_loop(
        model_base, (len(preview_labels),3,64,64),
        class_labels=preview_labels, device=device)
grid = vutils.make_grid(denorm(samples.cpu()), nrow=8, normalize=False)
plt.figure(figsize=(14,4)); plt.axis('off')
plt.title('Baseline DDPM — Generated Samples')
plt.imshow(grid.permute(1,2,0).numpy())
plt.savefig('../outputs/ddpm_baseline_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Grid Search over Hyper-parameters

In [ ]:
GRID = {
    'timesteps': [200, 500],
    'schedule' : ['cosine', 'linear'],
    'base_ch'  : [64],
    'lr'       : [1e-4, 2e-4],
}
configs = list(itertools.product(*GRID.values()))
keys    = list(GRID.keys())
print(f'Grid search: {len(configs)} configs x 30 epochs each')

gs_results = []
for cfg in configs:
    params = dict(zip(keys, cfg))
    print(f'  {params} ...', end=' ', flush=True)
    try:
        _, _, val_loss, _ = train_ddpm(**params, epochs=30, patience=5, verbose=False)
        params['val_loss'] = round(val_loss, 6)
        gs_results.append(params)
        print(f'val_loss={val_loss:.4f}')
    except Exception as e:
        print(f'FAILED: {e}')

gs_df = pd.DataFrame(gs_results).sort_values('val_loss')
print('\nTop configs:')
print(gs_df.head(5).to_string(index=False))

## 6. Final Training with Best Config

In [ ]:
best_cfg = gs_df.iloc[0].to_dict()
best_cfg_params = {
    'timesteps': int(best_cfg['timesteps']),
    'schedule' : best_cfg['schedule'],
    'base_ch'  : int(best_cfg['base_ch']),
    'lr'       : float(best_cfg['lr']),
}
print(f'Best config: {best_cfg_params}')
print('Training final DDPM (60 epochs, patience=10)...')
model_final, diff_final, final_val_loss, final_hist = train_ddpm(
    **best_cfg_params, epochs=60, patience=10, verbose=True)
print(f'Final DDPM best val loss: {final_val_loss:.4f}')

os.makedirs('../outputs/models', exist_ok=True)
torch.save(model_final.state_dict(), '../outputs/models/ddpm_best.pt')
with open('../outputs/ddpm_best_config.json','w') as f:
    json.dump({**best_cfg_params, 'val_loss': final_val_loss}, f, indent=2)
print('Model saved.')

In [ ]:
hist_df = pd.DataFrame(final_hist)
plt.figure(figsize=(8,4))
plt.plot(hist_df['epoch'], hist_df['train_loss'], label='Train')
plt.plot(hist_df['epoch'], hist_df['val_loss'],   label='Val')
plt.title('Final DDPM — Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
plt.savefig('../outputs/ddpm_final_loss.png', dpi=150); plt.show()

In [ ]:
model_final.eval()
preview_labels = torch.arange(min(16, NUM_CLASSES), device=device)
with torch.no_grad():
    samples = diff_final.p_sample_loop(
        model_final, (len(preview_labels),3,64,64),
        class_labels=preview_labels, device=device)
grid = vutils.make_grid(denorm(samples.cpu()), nrow=8, normalize=False)
plt.figure(figsize=(14,4)); plt.axis('off')
plt.title('Final DDPM — Generated Samples')
plt.imshow(grid.permute(1,2,0).numpy())
plt.savefig('../outputs/ddpm_final_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Generative Evaluation Metrics (SSIM, FID, IS)

In [ ]:
def compute_generative_metrics(model, diffusion, loader, n_samples=200):
    model.eval()
    ssim_scores = []
    fid_m = FrechetInceptionDistance(feature=64, normalize=True)
    is_m  = InceptionScore(feature='logits_softmax', normalize=True)
    real_list, gen_list, done = [], [], 0

    with torch.no_grad():
        for imgs, labels in loader:
            if done >= n_samples: break
            imgs = imgs.to(device); labels = labels.to(device)
            gen  = diffusion.p_sample_loop(model, imgs.shape, class_labels=labels, device=device)

            for r, g in zip(denorm(imgs.cpu()).permute(0,2,3,1).numpy(),
                            denorm(gen.cpu()).permute(0,2,3,1).numpy()):
                ssim_scores.append(ssim_metric(r, g, channel_axis=2, data_range=1.0))

            real_list.append((denorm(imgs.cpu())*255).byte())
            gen_list.append((denorm(gen.cpu())*255).byte())
            done += imgs.size(0)

    real_all = torch.cat(real_list)[:n_samples]
    gen_all  = torch.cat(gen_list)[:n_samples]
    fid_m.update(real_all, real=True); fid_m.update(gen_all, real=False)
    is_m.update(gen_all.float()/255.0)
    is_mean, is_std = is_m.compute()

    return {
        'ssim': float(np.mean(ssim_scores)),
        'fid':  float(fid_m.compute()),
        'is_mean': float(is_mean),
        'is_std':  float(is_std),
    }

print('Computing generative metrics...')
ddpm_metrics = compute_generative_metrics(model_final, diff_final, val_ld, n_samples=200)
print(f"SSIM : {ddpm_metrics['ssim']:.4f}")
print(f"FID  : {ddpm_metrics['fid']:.4f}")
print(f"IS   : {ddpm_metrics['is_mean']:.4f} +/- {ddpm_metrics['is_std']:.4f}")

## 8. Generate Augmented Dataset

In [ ]:
IMAGES_PER_CLASS = 75
OUT_DIR = '../outputs/generated_ddpm'
os.makedirs(OUT_DIR, exist_ok=True)
model_final.eval()
total_generated = 0

print(f'Generating {IMAGES_PER_CLASS} images per class ({NUM_CLASSES} classes)...')
for cls_idx, cls_name in tqdm(idx_to_class.items(), desc='Classes'):
    cls_dir = os.path.join(OUT_DIR, cls_name)
    os.makedirs(cls_dir, exist_ok=True)
    generated = 0
    while generated < IMAGES_PER_CLASS:
        n = min(8, IMAGES_PER_CLASS - generated)
        labels = torch.full((n,), cls_idx, device=device, dtype=torch.long)
        with torch.no_grad():
            imgs = diff_final.p_sample_loop(
                model_final, (n,3,64,64), class_labels=labels, device=device)
        imgs_np = (denorm(imgs.cpu()).permute(0,2,3,1).numpy() * 255).astype('uint8')
        for img_arr in imgs_np:
            Image.fromarray(img_arr).save(
                os.path.join(cls_dir, f'gen_{generated:04d}.png'))
            generated += 1
    total_generated += generated

print(f'Done! Total generated: {total_generated}')

## 9. Retrain Baseline CNN with DDPM Augmentation

In [ ]:
# Build augmented training set dataframe
aug_rows = []
for cls_idx, cls_name in idx_to_class.items():
    cls_dir = os.path.join(OUT_DIR, cls_name)
    if not os.path.isdir(cls_dir): continue
    for fname in sorted(os.listdir(cls_dir)):
        if fname.endswith('.png'):
            aug_rows.append({'filename': os.path.join(cls_dir, fname), 'label': cls_name})
aug_df = pd.DataFrame(aug_rows)
print(f'Generated images: {len(aug_df)}')

train_t, val_t = get_transforms()

class AbsPathDataset(data.Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.c2i = class_to_idx
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filename']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(self.c2i[row['label']], dtype=torch.long)

orig_ds  = ButterflyDataset(df=train_df, img_dir=train_dir, transform=train_t)
gen_ds   = AbsPathDataset(aug_df, class_to_idx, transform=train_t)
combo_ds = data.ConcatDataset([orig_ds, gen_ds])
print(f'Combined dataset: {len(combo_ds)} images')

combo_ld = data.DataLoader(combo_ds, batch_size=64, shuffle=True, num_workers=0)
val_cls_ds = ButterflyDataset(df=val_df, img_dir=train_dir, transform=val_t)
val_cls_ld = data.DataLoader(val_cls_ds, batch_size=64, shuffle=False, num_workers=0)

In [ ]:
def train_classifier(train_loader, val_loader, n_classes, epochs=60, patience=10):
    clf = BaselineCNN(num_classes=n_classes).to(device)
    opt = optim.Adam(clf.parameters(), lr=1e-3)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    criterion = nn.CrossEntropyLoss()
    best_acc, no_imp, best_state = 0, 0, None
    history = []

    for epoch in range(epochs):
        clf.train(); ep_loss = 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            loss = criterion(clf(imgs), lbls)
            loss.backward(); opt.step()
            ep_loss += loss.item()
        ep_loss /= len(train_loader)
        sched.step()

        clf.eval(); all_pred, all_true = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                preds = clf(imgs.to(device)).argmax(1).cpu()
                all_pred.extend(preds.tolist())
                all_true.extend(lbls.tolist())
        acc = accuracy_score(all_true, all_pred)
        f1  = f1_score(all_true, all_pred, average='weighted', zero_division=0)
        history.append({'epoch': epoch+1, 'loss': ep_loss, 'val_acc': acc, 'val_f1': f1})
        if (epoch+1) % 10 == 0:
            print(f'Epoch {epoch+1:3d} | loss={ep_loss:.4f} | acc={acc:.4f} | f1={f1:.4f}')

        if acc > best_acc:
            best_acc = acc; no_imp = 0
            best_state = {k: v.cpu().clone() for k,v in clf.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= patience: break

    clf.load_state_dict(best_state)
    return clf, best_acc, history

print('Training Baseline CNN with DDPM augmentation...')
clf_ddpm, best_acc_ddpm, hist_ddpm = train_classifier(combo_ld, val_cls_ld, NUM_CLASSES)

clf_ddpm.eval(); all_pred, all_true = [], []
with torch.no_grad():
    for imgs, lbls in val_cls_ld:
        preds = clf_ddpm(imgs.to(device)).argmax(1).cpu()
        all_pred.extend(preds.tolist()); all_true.extend(lbls.tolist())
ddpm_acc = accuracy_score(all_true, all_pred)
ddpm_f1  = f1_score(all_true, all_pred, average='weighted', zero_division=0)
print(f'DDPM Aug CNN | val_acc={ddpm_acc:.4f} | val_f1={ddpm_f1:.4f}')

## 10. Results & Comparison

In [ ]:
with open('../outputs/baseline_results.json') as f: base_res = json.load(f)
with open('../outputs/vae_aug_results.json')  as f: vae_res  = json.load(f)

ddpm_results = {
    'model': 'Baseline CNN + DDPM aug',
    'val_accuracy': ddpm_acc,
    'val_f1_weighted': ddpm_f1,
    'ddpm_config': best_cfg_params,
    'generative_metrics': ddpm_metrics,
}
with open('../outputs/ddpm_aug_results.json','w') as f:
    json.dump(ddpm_results, f, indent=2)

comparison = pd.DataFrame([
    {'Model': 'Baseline CNN (no aug)',
     'Val Accuracy': round(base_res['val_accuracy'],4),
     'Val F1':       round(base_res['val_f1_weighted'],4),
     'SSIM': '-', 'FID': '-', 'IS': '-'},
    {'Model': 'Baseline CNN + VAE aug',
     'Val Accuracy': round(vae_res['val_accuracy'],4),
     'Val F1':       round(vae_res['val_f1_weighted'],4),
     'SSIM': f"{vae_res['generative_metrics']['ssim']:.4f}",
     'FID':  f"{vae_res['generative_metrics']['fid']:.4f}",
     'IS':   f"{vae_res['generative_metrics']['is_mean']:.4f}"},
    {'Model': 'Baseline CNN + DDPM aug',
     'Val Accuracy': round(ddpm_acc,4),
     'Val F1':       round(ddpm_f1,4),
     'SSIM': f"{ddpm_metrics['ssim']:.4f}",
     'FID':  f"{ddpm_metrics['fid']:.4f}",
     'IS':   f"{ddpm_metrics['is_mean']:.4f}"},
])
print(comparison.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
model_names = ['Baseline\n(no aug)', 'VAE aug', 'DDPM aug']
accs = [base_res['val_accuracy'], vae_res['val_accuracy'], ddpm_acc]
f1s  = [base_res['val_f1_weighted'], vae_res['val_f1_weighted'], ddpm_f1]
colors = ['#4e79a7', '#f28e2b', '#e15759']

ax1.bar(model_names, accs, color=colors)
ax1.set_title('Validation Accuracy'); ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, max(accs)*1.25)
for i,v in enumerate(accs): ax1.text(i, v+0.005, f'{v:.3f}', ha='center', fontweight='bold')

ax2.bar(model_names, f1s, color=colors)
ax2.set_title('Validation F1 (weighted)'); ax2.set_ylabel('F1 Score')
ax2.set_ylim(0, max(f1s)*1.25)
for i,v in enumerate(f1s): ax2.text(i, v+0.005, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Generative Augmentation — Model Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/ddpm_comparison.png', dpi=150); plt.show()

print(f"\nBaseline accuracy : {base_res['val_accuracy']:.4f}")
print(f"VAE aug accuracy  : {vae_res['val_accuracy']:.4f}  (Delta={vae_res['val_accuracy']-base_res['val_accuracy']:+.4f})")
print(f"DDPM aug accuracy : {ddpm_acc:.4f}  (Delta={ddpm_acc-base_res['val_accuracy']:+.4f})")